In [0]:
# ==========================================================
# 01. EXPLORACIÓN BRONZE - CATÁLOGO DE PRODUCTOS
# Proyecto: ElectroCasa
# Capa: Silver
# ==========================================================

CATALOGO = "proyecto_final"

TABLA_ORIGEN = f"{CATALOGO}.bronze.catalogo"
TABLA_DESTINO = f"{CATALOGO}.silver.catalogo"
TABLA_CUARENTENA = f"{CATALOGO}.silver.cuarentena"

# Leer tabla Bronze
df_bronze = spark.table(TABLA_ORIGEN)

print("===== CONFIGURACIÓN =====")
print("ORIGEN:", TABLA_ORIGEN)
print("DESTINO SILVER:", TABLA_DESTINO)
print("CUARENTENA:", TABLA_CUARENTENA)

print("\n===== ESQUEMA BRONZE =====")
df_bronze.printSchema()

print("\n===== TOTAL DE REGISTROS =====")
print(df_bronze.count())

print("\n===== MUESTRA DE DATOS =====")
display(df_bronze.limit(10))

In [0]:
%sql
SELECT
    archivo_origen,
    COUNT(*) AS total_registros,
    COUNT(DISTINCT producto_id) AS productos_distintos,
    COUNT(*) - COUNT(DISTINCT producto_id) AS registros_adicionales
FROM proyecto_final.bronze.catalogo
GROUP BY archivo_origen
ORDER BY archivo_origen;

In [0]:
%sql
SELECT
    archivo_origen,
    COUNT(*) AS total_registros,
    SUM(
        CASE
            WHEN producto_id IS NULL
              OR TRIM(producto_id) = ''
            THEN 1
            ELSE 0
        END
    ) AS ids_invalidos,
    COUNT(DISTINCT producto_id) AS productos_distintos
FROM proyecto_final.bronze.catalogo
GROUP BY archivo_origen
ORDER BY archivo_origen;

In [0]:
%sql
SELECT
    archivo_origen,
    COUNT(*) AS total_registros,

    SUM(
        CASE
            WHEN precio_lista IS NULL
              OR TRIM(precio_lista) = ''
            THEN 1 ELSE 0
        END
    ) AS precios_nulos,

    SUM(
        CASE
            WHEN TRY_CAST(
                REGEXP_REPLACE(precio_lista, '[^0-9.-]', '')
                AS DECIMAL(12,2)
            ) <= 0
            THEN 1 ELSE 0
        END
    ) AS precios_no_positivos,

    SUM(
        CASE
            WHEN precio_lista IS NOT NULL
             AND TRIM(precio_lista) <> ''
             AND TRY_CAST(
                 REGEXP_REPLACE(precio_lista, '[^0-9.-]', '')
                 AS DECIMAL(12,2)
             ) IS NULL
            THEN 1 ELSE 0
        END
    ) AS precios_no_convertibles

FROM proyecto_final.bronze.catalogo
GROUP BY archivo_origen
ORDER BY archivo_origen;

In [0]:
%sql
WITH productos AS (
    SELECT
        producto_id,
        archivo_origen,
        precio_lista,
        TRY_CAST(
            REGEXP_REPLACE(
                precio_lista,
                '[^0-9.-]',
                ''
            ) AS DECIMAL(12,2)
        ) AS precio_decimal
    FROM proyecto_final.bronze.catalogo
),

marzo AS (
    SELECT
        producto_id,
        MAX(precio_decimal) AS precio_marzo
    FROM productos
    WHERE archivo_origen LIKE '%2026-03-01%'
      AND precio_decimal > 0
    GROUP BY producto_id
),

agosto AS (
    SELECT
        producto_id,
        precio_lista,
        precio_decimal
    FROM productos
    WHERE archivo_origen LIKE '%2026-08-01%'
      AND (
          precio_decimal IS NULL
          OR precio_decimal <= 0
      )
)

SELECT
    COUNT(*) AS productos_invalidos_agosto,

    SUM(
        CASE
            WHEN m.producto_id IS NOT NULL
            THEN 1 ELSE 0
        END
    ) AS con_precio_valido_marzo,

    SUM(
        CASE
            WHEN m.producto_id IS NULL
            THEN 1 ELSE 0
        END
    ) AS sin_precio_valido_marzo

FROM agosto a
LEFT JOIN marzo m
    ON a.producto_id = m.producto_id;

In [0]:
%sql
SELECT
    categoria,
    COUNT(*) AS total_registros
FROM proyecto_final.bronze.catalogo
GROUP BY categoria
ORDER BY categoria;

In [0]:
%sql
SELECT
    archivo_origen,
    COUNT(*) AS total_registros,

    SUM(
        CASE
            WHEN nombre_producto IS NULL
              OR TRIM(nombre_producto) = ''
            THEN 1 ELSE 0
        END
    ) AS nombres_invalidos,

    SUM(
        CASE
            WHEN marca IS NULL
              OR TRIM(marca) = ''
            THEN 1 ELSE 0
        END
    ) AS marcas_nulas,

    SUM(
        CASE
            WHEN activo IS NULL
            THEN 1 ELSE 0
        END
    ) AS estados_nulos,

    SUM(
        CASE
            WHEN activo = TRUE
            THEN 1 ELSE 0
        END
    ) AS productos_activos,

    SUM(
        CASE
            WHEN activo = FALSE
            THEN 1 ELSE 0
        END
    ) AS productos_inactivos

FROM proyecto_final.bronze.catalogo
GROUP BY archivo_origen
ORDER BY archivo_origen;

In [0]:
%sql
WITH marzo AS (
    SELECT DISTINCT
        producto_id
    FROM proyecto_final.bronze.catalogo
    WHERE archivo_origen LIKE '%2026-03-01%'
),

agosto AS (
    SELECT DISTINCT
        producto_id
    FROM proyecto_final.bronze.catalogo
    WHERE archivo_origen LIKE '%2026-08-01%'
)

SELECT
    COUNT(CASE
        WHEN m.producto_id IS NOT NULL
         AND a.producto_id IS NOT NULL
        THEN 1
    END) AS productos_ambos_snapshots,

    COUNT(CASE
        WHEN m.producto_id IS NOT NULL
         AND a.producto_id IS NULL
        THEN 1
    END) AS productos_solo_marzo,

    COUNT(CASE
        WHEN m.producto_id IS NULL
         AND a.producto_id IS NOT NULL
        THEN 1
    END) AS productos_nuevos_agosto

FROM marzo m
FULL OUTER JOIN agosto a
    ON m.producto_id = a.producto_id;

In [0]:
# ==========================================================
# 09. PREPARACIÓN DE DATOS - CATÁLOGO
# Proyecto: ElectroCasa
# Capa: Silver
# ==========================================================

from pyspark.sql.functions import (
    col,
    to_json,
    struct,
    regexp_extract,
    to_date,
    max as spark_max
)

# Leer Bronze
df_bronze = spark.table("proyecto_final.bronze.catalogo")

# Conservar el registro original para cuarentena
df_preparado = (
    df_bronze
    .withColumn(
        "_datos_originales",
        to_json(
            struct(*[col(c) for c in df_bronze.columns])
        )
    )
    .withColumn(
        "fecha_snapshot",
        to_date(
            regexp_extract(
                col("archivo_origen"),
                r"(\d{4}-\d{2}-\d{2})",
                1
            ),
            "yyyy-MM-dd"
        )
    )
)

# Identificar el snapshot más reciente
fecha_ultimo_snapshot = (
    df_preparado
    .agg(spark_max("fecha_snapshot").alias("fecha_maxima"))
    .first()["fecha_maxima"]
)

# Seleccionar únicamente el último snapshot
df_vigente = df_preparado.filter(
    col("fecha_snapshot") == fecha_ultimo_snapshot
)

print("===== PREPARACIÓN DEL CATÁLOGO =====")
print("Registros Bronze:", df_bronze.count())
print("Último snapshot:", fecha_ultimo_snapshot)
print("Registros del snapshot vigente:", df_vigente.count())

print("\nESQUEMA PREPARADO:")
df_vigente.printSchema()

print("\nMUESTRA:")
display(df_vigente.limit(10))

In [0]:
# ==========================================================
# 10. NORMALIZACIÓN DE CATÁLOGO
# Proyecto: ElectroCasa
# Capa: Silver
# ==========================================================

from pyspark.sql.functions import (
    col,
    trim,
    lower,
    regexp_replace
)

# 1. Eliminar espacios en los campos de texto
df_limpio = (
    df_vigente
    .withColumn("producto_id", trim(col("producto_id")))
    .withColumn("nombre_producto", trim(col("nombre_producto")))
    .withColumn("categoria", trim(col("categoria")))
    .withColumn("marca", trim(col("marca")))
)

# 2. Normalizar categorías
df_normalizado = (
    df_limpio
    .withColumn("categoria", lower(col("categoria")))
    .withColumn("categoria", regexp_replace(col("categoria"), "á", "a"))
    .withColumn("categoria", regexp_replace(col("categoria"), "é", "e"))
    .withColumn("categoria", regexp_replace(col("categoria"), "í", "i"))
    .withColumn("categoria", regexp_replace(col("categoria"), "ó", "o"))
    .withColumn("categoria", regexp_replace(col("categoria"), "ú", "u"))
    .withColumn("categoria", regexp_replace(col("categoria"), r"\s+", "_"))
)

# 3. Convertir precios a DECIMAL(12,2)
df_normalizado = (
    df_normalizado
    .withColumn(
        "precio_lista",
        regexp_replace(col("precio_lista"), ",", "")
    )
    .withColumn(
        "precio_lista",
        regexp_replace(col("precio_lista"), r"[^0-9.-]", "")
    )
    .withColumn(
        "precio_lista",
        col("precio_lista").try_cast("decimal(12,2)")
    )
)

# 4. Verificar resultados
print("===== NORMALIZACIÓN DE CATÁLOGO =====")
print("Total normalizados:", df_normalizado.count())

print("\nCATEGORÍAS NORMALIZADAS:")
display(
    df_normalizado
    .groupBy("categoria")
    .count()
    .orderBy("categoria")
)

print("\nESQUEMA RESULTANTE:")
df_normalizado.printSchema()

In [0]:
# ==========================================================
# 11. VALIDACIÓN DE PRECIOS Y ESTADO
# Proyecto: ElectroCasa
# Capa: Silver
# ==========================================================

from pyspark.sql.functions import (
    col,
    count,
    sum as spark_sum,
    when
)

print("===== VALIDACIÓN DE PRECIOS POR ESTADO =====")

display(
    df_normalizado
    .groupBy("activo")
    .agg(
        count("*").alias("total_productos"),

        spark_sum(
            when(
                col("precio_lista").isNull(),
                1
            ).otherwise(0)
        ).alias("precios_nulos"),

        spark_sum(
            when(
                col("precio_lista") <= 0,
                1
            ).otherwise(0)
        ).alias("precios_no_positivos"),

        spark_sum(
            when(
                col("precio_lista") > 0,
                1
            ).otherwise(0)
        ).alias("precios_validos")
    )
    .orderBy("activo")
)

In [0]:
# ==========================================================
# 12. REGLAS DE CALIDAD - CATÁLOGO
# Proyecto: ElectroCasa
# Capa: Silver
# ==========================================================

from pyspark.sql.functions import (
    col,
    trim,
    when,
    lit,
    concat_ws
)

CATEGORIAS_VALIDAS = [
    "linea_blanca",
    "cocina",
    "electronica",
    "climatizacion",
    "entretenimiento"
]

df_validado = (
    df_normalizado
    .withColumn(
        "motivo_rechazo",
        concat_ws(
            ", ",

            when(
                col("producto_id").isNull() |
                (trim(col("producto_id")) == ""),
                lit("PRODUCTO_ID_INVALIDO")
            ),

            when(
                col("nombre_producto").isNull() |
                (trim(col("nombre_producto")) == ""),
                lit("NOMBRE_PRODUCTO_INVALIDO")
            ),

            when(
                col("categoria").isNull() |
                (~col("categoria").isin(CATEGORIAS_VALIDAS)),
                lit("CATEGORIA_INVALIDA")
            ),

            when(
                col("precio_lista").isNull(),
                lit("PRECIO_NULO_O_NO_CONVERTIBLE")
            ),

            when(
                col("precio_lista") <= 0,
                lit("PRECIO_NO_POSITIVO")
            ),

            when(
                col("activo").isNull(),
                lit("ESTADO_INVALIDO")
            )
        )
    )
)

# Separar registros válidos
df_validos = df_validado.filter(
    col("motivo_rechazo") == ""
).drop("motivo_rechazo")

# Separar registros rechazados
df_rechazados = df_validado.filter(
    col("motivo_rechazo") != ""
)

print("===== RESULTADOS DE CALIDAD =====")
print("Total normalizados:", df_normalizado.count())
print("Registros válidos:", df_validos.count())
print("Registros rechazados:", df_rechazados.count())

print("\nMOTIVOS DE RECHAZO:")

display(
    df_rechazados
    .groupBy("motivo_rechazo")
    .count()
    .orderBy("motivo_rechazo")
)

print("\nMUESTRA DE REGISTROS RECHAZADOS:")

display(
    df_rechazados.select(
        "producto_id",
        "nombre_producto",
        "precio_lista",
        "activo",
        "motivo_rechazo"
    ).limit(10)
)

In [0]:
# ==========================================================
# 13. CONTROL DE DUPLICADOS - CATÁLOGO
# Proyecto: ElectroCasa
# Capa: Silver
# ==========================================================

from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col,
    row_number,
    lit
)

# Criterio de selección determinista
ventana_productos = (
    Window
    .partitionBy("producto_id")
    .orderBy(
        col("fecha_snapshot").desc(),
        col("fecha_ingesta").desc(),
        col("precio_lista").desc(),
        col("nombre_producto").asc(),
        col("categoria").asc(),
        col("marca").asc_nulls_last(),
        col("archivo_origen").asc()
    )
)

# Numerar registros por producto
df_numerado = df_validos.withColumn(
    "_numero_fila",
    row_number().over(ventana_productos)
)

# Conservar un registro por producto
df_silver = (
    df_numerado
    .filter(col("_numero_fila") == 1)
    .drop("_numero_fila")
)

# Separar duplicados excedentes
df_duplicados = (
    df_numerado
    .filter(col("_numero_fila") > 1)
    .drop("_numero_fila")
    .withColumn(
        "motivo_rechazo",
        lit("PRODUCTO_DUPLICADO")
    )
)

# Unir todos los registros destinados a cuarentena
df_cuarentena_catalogo = df_rechazados.unionByName(
    df_duplicados
)

print("===== CONTROL DE DUPLICADOS =====")
print("Registros válidos antes de deduplicar:", df_validos.count())
print("Productos únicos para Silver:", df_silver.count())
print("Registros duplicados:", df_duplicados.count())
print("Total para cuarentena:", df_cuarentena_catalogo.count())

print("\nPRODUCTOS DUPLICADOS:")
display(df_duplicados)

In [0]:
# ==========================================================
# 14. PREPARACIÓN DE TABLA SILVER - CATÁLOGO
# Proyecto: ElectroCasa
# ==========================================================

from pyspark.sql.functions import col

TABLA_DESTINO = "proyecto_final.silver.catalogo"

# Seleccionar las columnas del contrato
df_silver_final = df_silver.select(
    "producto_id",
    "nombre_producto",
    "categoria",
    "marca",
    "precio_lista",
    "activo",
    "fecha_snapshot"
)

print("===== CONTROL PREVIO A SILVER =====")

total_vigente = df_vigente.count()
total_validos = df_silver_final.count()
total_rechazados = df_cuarentena_catalogo.count()

print("Registros del snapshot vigente:", total_vigente)
print("Registros válidos para Silver:", total_validos)
print("Registros para cuarentena:", total_rechazados)

# Verificar que todos los registros tengan un destino
assert total_vigente == total_validos + total_rechazados, \
    "ERROR: No coincide el total de registros."

# Verificar que existan registros válidos
assert total_validos > 0, \
    "ERROR: No existen registros válidos."

print("\nVALIDACIÓN DE CONTEOS CORRECTA")

print("\nESQUEMA FINAL DE SILVER:")
df_silver_final.printSchema()

print("\nMUESTRA DE PRODUCTOS:")
display(df_silver_final.limit(10))

In [0]:
%sql
SHOW TABLES IN proyecto_final.silver
LIKE 'catalogo';

In [0]:
# ==========================================================
# 16. ESCRITURA SILVER - CATÁLOGO
# Proyecto: ElectroCasa
# ==========================================================

TABLA_DESTINO = "proyecto_final.silver.catalogo"

print("===== ESCRITURA SILVER =====")
print("Tabla destino:", TABLA_DESTINO)
print("Registros para Silver:", df_silver_final.count())

# Evitar escribir sobre una tabla existente
if spark.catalog.tableExists(TABLA_DESTINO):
    raise Exception(
        f"La tabla {TABLA_DESTINO} ya existe. "
        "Verificar antes de volver a escribir."
    )

# Crear tabla Delta
(
    df_silver_final.write
    .format("delta")
    .mode("append")
    .saveAsTable(TABLA_DESTINO)
)

print("\nTabla Silver creada correctamente.")
print("DESTINO:", TABLA_DESTINO)

In [0]:
%sql
SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT producto_id) AS productos_distintos,
    COUNT(DISTINCT categoria) AS categorias_distintas,

    SUM(
        CASE WHEN activo = TRUE THEN 1 ELSE 0 END
    ) AS productos_activos,

    SUM(
        CASE WHEN activo = FALSE THEN 1 ELSE 0 END
    ) AS productos_inactivos,

    SUM(
        CASE
            WHEN precio_lista IS NULL OR precio_lista <= 0
            THEN 1 ELSE 0
        END
    ) AS precios_invalidos,

    MIN(fecha_snapshot) AS fecha_minima,
    MAX(fecha_snapshot) AS fecha_maxima

FROM proyecto_final.silver.catalogo;

In [0]:
%sql
SELECT
    fuente,
    COUNT(*) AS total_rechazados
FROM proyecto_final.silver.cuarentena
GROUP BY fuente
ORDER BY fuente;

In [0]:
# ==========================================================
# 19. PREPARACIÓN DE CUARENTENA - CATÁLOGO
# Proyecto: ElectroCasa
# ==========================================================

from pyspark.sql.functions import (
    col,
    lit,
    current_timestamp
)

TABLA_CUARENTENA = "proyecto_final.silver.cuarentena"

df_cuarentena_final = (
    df_cuarentena_catalogo
    .select(
        lit("catalogo").alias("fuente"),
        col("producto_id").alias("clave_registro"),
        col("motivo_rechazo"),
        current_timestamp().alias("fecha_rechazo"),
        col("_datos_originales").alias("datos_originales"),
        col("archivo_origen")
    )
)

print("===== PREPARACIÓN DE CUARENTENA =====")
print("Destino:", TABLA_CUARENTENA)
print("Registros preparados:", df_cuarentena_final.count())

print("\nESQUEMA:")
df_cuarentena_final.printSchema()

print("\nMUESTRA DE REGISTROS:")
display(df_cuarentena_final.limit(10))

In [0]:
# ==========================================================
# 20. ESCRITURA EN CUARENTENA - CATÁLOGO
# Proyecto: ElectroCasa
# ==========================================================

TABLA_CUARENTENA = "proyecto_final.silver.cuarentena"

# Comprobar si ya existen rechazos del catálogo
total_existentes = (
    spark.table(TABLA_CUARENTENA)
    .filter("fuente = 'catalogo'")
    .count()
)

total_preparados = df_cuarentena_final.count()

print("===== CONTROL DE CUARENTENA =====")
print("Registros existentes de catálogo:", total_existentes)
print("Registros preparados:", total_preparados)

# Evitar una carga duplicada
if total_existentes > 0:
    print("\nYa existen registros de catálogo en cuarentena.")
    print("No se realizará una nueva inserción.")
else:
    (
        df_cuarentena_final.write
        .format("delta")
        .mode("append")
        .saveAsTable(TABLA_CUARENTENA)
    )

    print("\nRegistros rechazados almacenados correctamente.")
    print("Destino:", TABLA_CUARENTENA)

In [0]:
%sql
SELECT
    fuente,
    clave_registro,
    motivo_rechazo,
    fecha_rechazo,
    datos_originales,
    archivo_origen
FROM proyecto_final.silver.cuarentena
WHERE fuente = 'catalogo'
ORDER BY motivo_rechazo, clave_registro;

In [0]:
%sql
SELECT
    fuente,
    motivo_rechazo,
    COUNT(*) AS total_rechazados
FROM proyecto_final.silver.cuarentena
WHERE fuente = 'catalogo'
GROUP BY fuente, motivo_rechazo
ORDER BY motivo_rechazo;

In [0]:
%sql
SELECT
    (SELECT COUNT(*)
     FROM proyecto_final.bronze.catalogo
     WHERE archivo_origen = 'catalogo_productos_2026-08-01.json'
    ) AS registros_snapshot_agosto,

    (SELECT COUNT(*)
     FROM proyecto_final.silver.catalogo
    ) AS registros_silver,

    (SELECT COUNT(*)
     FROM proyecto_final.silver.cuarentena
     WHERE fuente = 'catalogo'
    ) AS registros_cuarentena,

    (SELECT COUNT(*)
     FROM proyecto_final.silver.catalogo
    ) +
    (SELECT COUNT(*)
     FROM proyecto_final.silver.cuarentena
     WHERE fuente = 'catalogo'
    ) AS total_conciliado;